---
title: "Trotterized Hubbard Model"
title-block-banner: true
abstract: |
  This article evaluates novel approaches to do
  some really important things.
---

todo: Check neel state set up

Execute summary of whole thing, then of a section

# Motivation for Trotterization

Why QSVT with QROM vs Trooterization

Hubbard is local and structured
QSVT/QROM is especially powerful when the Hamiltonian is given as a large irregular table: But the Hubbard model on a regular lattice is not an arbitrary table. It has simple repeated structure:

- nearest-neighbor hopping,
- osite interaction,
- mostly uniform coefficients,
- sparse local geometry.


In [5]:
import logging

import numpy as np
import pennylane as qp
import pennylane.estimator as qre
from pennylane.resource import SpectralNormError
import tqdm

logging.basicConfig(level=logging.INFO)

np.random.seed(4)
qp.numpy.random.seed(4)

# Setup {#sec-setup}

We create a $2 \times 2 \ times 1$ cube, with 4 sites, and 8 qubits. The notion is 

| Qubit | Spin orbital |
|-------|--------------|
| 0     | site 0 ↑     |
| 1     | site 0 ↓     |
| 2     | site 1 ↑     |
| 3     | site 1 ↓     |
| 4     | site 2 ↑     |
| 5     | site 2 ↓     |
| 6     | site 3 ↑     |
| 7     | site 3 ↓     |


In [6]:
#| code-fold: true
t = 1.0
U = 4.0
time = 2.4                  # The time of evolution (t in exp(iHt))
n_steps = int(time * 25)    # number of Trotter steps
n_cells = [2, 2, 1]
orders = [1, 2, 4]
time = 5
H = qp.spin.fermi_hubbard(
    lattice="cubic",
    n_cells=n_cells,
    hopping=t,
    coulomb=U,
    boundary_condition=False,
    mapping="jordan_wigner",
)
n_qubits = H.num_wires

This link is interesting https://fermi-hubbard-commutators.readthxxedocs.io/en/latest/commutator_bounds.html

## Comparison
How badly do standard Trotter error bounds overestimate cost for specific observables in Hubbard, and what's the practical implication for resource estimates?"


1. Calculate the actual $\exp(-itH)$
2. Trotterize $H$
3. Calculate error
    1. Calculate the Spectral error
4. Calculate cost
    1. Define a circuit
    2. Estimate the number of gates needed to reach such error level
5. Plot (3) and (4)

However, the idea is about how to not actually have to run the simulation and still able to estimate the error.

## Worst-case vs. typical-case
The spectral norm $\lVert {U_{exact} - U_{trotter}}\rVert_2$ measures the maximum possible deviation over all possible input states in the entire Hilbert space.

However in common application, these relevant states (e.g, low-energy excited states with specific symmetries) is in a much smaller space. Therefore it is not the best interest to tune the Trotter using the Spectral Norm.

The next natural question is for the Hubbard model, the most relavant states are? In this example we use half-filled Neel state ($\ket{0101...01}$), which is standard for studying Mott insulators.

Haar-random states are commonly uses in quantum algorithm analysis because if its uniformly random over the entire Hilbert space. However have no structure relevant to the lattice, fermions, or interactions. Hubbard physics is about local correlations, hopping, double occupancy penalties, and half-filling.

Compute grouping here  https://docs.pennylane.ai/en/stable/code/api/pennylane.ops.op_math.LinearCombination.html#pennylane.ops.op_math.LinearCombination.compute_grouping

In [7]:
#H.compute_grouping()  # compute the qubit-wise commuting groups!

# resources_exec = qre.estimate(executable_circuit)(grouped_hamiltonian, num_steps, order)

# resources_with_grouping = qre.estimate(
#     qre.TrotterPauli(kitaev_H_with_grouping, num_steps, order)
# )

# res = qre.estimate(circuit)(kitaev_H_with_grouping, num_steps, order)

### Spectral Norm

Why makes sense, perhaps same base with true exp matrix, but different eigenvectors making it error?

In [8]:
exact_op = qp.exp(H, 1j * time)
spectral_error = {}

for order in tqdm.tqdm(orders):
    approx_op = qp.TrotterProduct(H, n=n_steps, time=time, order=order)
    error = SpectralNormError.get_error(exact_op, approx_op)
    spectral_error[order] = error

print("\n".join([f"Order: {k}, Spectral error: {v:5f}" for k, v in spectral_error.items()]))

Order: 1, Spectral error: 0.279030
Order: 2, Spectral error: 0.083633
Order: 4, Spectral error: 0.000116


### Childs Method

What can we do better than Spectral Norm?

Calculating spectral norm is expensive because it requires diagonalizing the operators.

In [18]:
for order in tqdm.tqdm(orders):
    op = qp.TrotterProduct(H, n=n_steps, time=time, order=order)

    one_norm_error_bound = op.error(method="one-norm-bound")
    commutator_error_bound = op.error(method="commutator-bound")

    print(f"Order {order}")
    print(f"One-norm bound: {one_norm_error_bound}")
    print(f"Commutator bound: {commutator_error_bound}")

one-norm bound:    SpectralNormError(720.0)
commutator bound:  SpectralNormError(45.925925925925924)


Now that we already have an operator level error, we can move on to resource estimation

## State initialization

Neel state ($\ket{0101...01}$) is the standard state for studying ferromagnetics problem. Using the notation in @sec-introduction, we deduce that

In [10]:
def prepare_neel_state(wires):
    """Néel state: alternating up/down on bipartite lattice"""
    for site in range(wires // 2):
        if site % 2 == 0:
            qp.PauliX(wires=2 * site)
        else:
            qp.PauliX(wires=2 * site + 1)

In [11]:
dev = qp.device("default.qubit")


@qp.qnode(dev)
def exact_circ(H, t, wires):
    """
    Simluate exact evolution
    """
    prepare_neel_state(wires)
    qp.exp(1j*t*H)
    return qp.state()


@qp.qnode(dev)
def trotter_circ(H, num_steps, t, wires, order=1):
    prepare_neel_state(wires)
    qp.TrotterProduct(H, n=num_steps, time=t, order=order)
    return qp.state()

In [12]:
resource_est = {}
for order in orders:
    resources_exec = qre.estimate(trotter_circ)(H, n_steps, time, n_qubits, order=order)
    resource_est[order] = resources_exec.gate_counts

In [13]:
resource_est

{1: defaultdict(int,
             {'X': 4,
              'Hadamard': 3840,
              'S': 1920,
              'Z': 960,
              'T': 73920,
              'CNOT': 6240}),
 2: defaultdict(int,
             {'X': 4,
              'Hadamard': 7444,
              'S': 3604,
              'Z': 1802,
              'T': 142604,
              'CNOT': 12124}),
 4: defaultdict(int,
             {'X': 4,
              'Hadamard': 37204,
              'S': 18004,
              'Z': 9002,
              'T': 712844,
              'CNOT': 60604})}

In [14]:
exact_state = exact_circ(H, time, n_qubits)
error_dicts = {}
state_fidelities = {}

for order in orders[:2]:
    trotter_state = trotter_circ(H, n_steps, time, n_qubits, order=order)
    state_fidelities[order] = qp.math.fidelity_statevector(exact_state, trotter_state)
    error_dicts[order] = qp.resource.algo_error(trotter_circ)(H, n_steps, time, n_qubits, order=order)

In [15]:
state_fidelities

{1: np.float64(0.9789934391057257), 2: np.float64(0.9990910458100898)}

In [16]:
error_dicts

{1: {'SpectralNormError': SpectralNormError(60.0)},
 2: {'SpectralNormError': SpectralNormError(45.925925925925924)}}

We care more about state fidelity 

# Hubbard specialized method

I expect a lot of clever commutator hack, and Hubbard model explou

## Business output

If we increase the order based on the Spectral norm estimation, then the number of gates vs accuracy looks like this

But if we increase the order based on the state fidelity, then the number of gates vs accuracy looks like this




# Goal

- Practical resource estimation for early-to-intermediate fault-tolerant quantum simulation of strongly correlated materials. Choose 3D Hubbard model using PennyLane's Labs estimator
- Give systematic, empirical comparison of worst-case vs. observable-specific Trotter error bounds
- Able to distinguish asymptotic theory from realistic hardware constraints;
- Capable of producing technically honest estimates that are useful for strategic decision-making inside an FTQC organization.

https://pennylane.ai/demos/tutorial_estimator_hamiltonian_simulation_gqsp: Why does this increase with time